In [0]:
# =============================================================================
# Phase 4.2 — Flight–Weather Join Pipeline
# =============================================================================
# What this notebook does (in order):
#   1. Reads and deduplicates the raw DOT flights parquet
#   2. Builds a reference table of unique airports with coordinates
#   3. Reads the pre-processed NOAA weather parquet + stations table
#   4. Finds the 5 closest weather stations to each airport (haversine)
#   5. Engineers the prediction-time window for each flight
#   6. Left-joins each flight to weather observations in its 3-hour window
#   7. Aggregates weather observations → one row per flight
#   8. [NEW] Adds graph-based features: PageRank, betweenness, cascade delay
#   9. Checkpoints the joined result to DBFS
# =============================================================================

# 0. Imports

In [0]:
import os
import networkx as nx
import pandas as pd
 
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

# 1. Setup

In [0]:
# Mount info for this class: Source for the original datasets (read access only, no writing)!
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

path,name,size,modificationTime
dbfs:/mnt/mids-w261/Data/,Data/,0,1775963472647
dbfs:/mnt/mids-w261/HW5/,HW5/,0,1775963472647
dbfs:/mnt/mids-w261/OTPW_12M/,OTPW_12M/,0,1775963472647
dbfs:/mnt/mids-w261/OTPW_1D_CSV/,OTPW_1D_CSV/,0,1775963472647
dbfs:/mnt/mids-w261/OTPW_36M/,OTPW_36M/,0,1775963472647
dbfs:/mnt/mids-w261/OTPW_3M/,OTPW_3M/,0,1775963472647
dbfs:/mnt/mids-w261/OTPW_3M_2015.csv,OTPW_3M_2015.csv,1500620247,1741625185000
dbfs:/mnt/mids-w261/OTPW_3M_2015_delta/,OTPW_3M_2015_delta/,0,1775963472647
dbfs:/mnt/mids-w261/OTPW_60M/,OTPW_60M/,0,1775963472647
dbfs:/mnt/mids-w261/OTPW_60M_Backup/,OTPW_60M_Backup/,0,1775963472647


In [0]:
display(dbutils.fs.ls("dbfs:/mnt/mids-w261/datasets_final_project_2022/"))

path,name,size,modificationTime
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data/,parquet_airlines_data/,0,1775963488403
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_1y/,parquet_airlines_data_1y/,0,1775963488403
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/,parquet_airlines_data_3m/,0,1775963488404
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_6m/,parquet_airlines_data_6m/,0,1775963488404
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/,parquet_weather_data/,0,1775963488404
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_1y/,parquet_weather_data_1y/,0,1775963488404
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_3m/,parquet_weather_data_3m/,0,1775963488404
dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data_6m/,parquet_weather_data_6m/,0,1775963488404
dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/,stations_data/,0,1775963488404


In [0]:
DATA_DIR          = "dbfs:/mnt/mids-w261/"
FLIGHTS_PATH      = DATA_DIR + "datasets_final_project_2022/parquet_airlines_data/"
WEATHER_PATH      = "dbfs:/student-groups/Group_01_02/kenzie_full_weather1.parquet"
STATIONS_PATH     = DATA_DIR + "datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/"
AIRPORT_CODES_PATH = DATA_DIR + "airport-codes_csv.csv"
MERGED_PATH       = "dbfs:/student-groups/Group_01_02/kenzie_full_df_joined.parquet"
 
# Only include domestic US territories — filters out international codeshares
# that appear in the DOT data but lack matching NOAA stations
YEAR_MAX          = 2020
RELEVANT_COUNTRIES = ["US", "PR", "VI", "GU", "AS", "MP"]
 
# ── Column lists ──────────────────────────────────────────────────────────────
 
# Raw NOAA hourly numeric fields — averaged across the weather window
NUMERIC_WEATHER_COLS = [
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyRelativeHumidity", "HourlyPrecipitation",
    "HourlyVisibility", "HourlyWindSpeed", "HourlyWindDirection",
    "HourlyWindGustSpeed", "HourlySeaLevelPressure", "HourlyStationPressure",
    "HourlyAltimeterSetting", "HourlyWetBulbTemperature", "HourlyPressureChange",
]
 
# Engineered weather features computed in Phase 3.1 — also averaged
NUMERIC_ENGINEERED_COLS = [
    "feat_dew_point_spread", "feat_wet_bulb_depression", "feat_oat_celsius",
    "feat_isa_temp_celsius", "feat_isa_deviation", "feat_elevation_feet",
    "feat_pressure_altitude", "feat_density_altitude", "feat_altimeter_deviation",
    "feat_pressure_change_magnitude", "HourlyWindDirection_Sin",
    "HourlyWindDirection_Cos", "feat_ceiling_hundreds_ft",
]
 
# Binary flags — take the most recent non-null value in the window instead of avg,
# because these are 0/1 indicators and averaging gives fractional values
# that are harder to interpret (e.g. 0.6 means "fog reported 60% of window")
BINARY_ENGINEERED_COLS = [
    "feat_fog_risk", "feat_is_freezing", "feat_near_freezing",
    "feat_rapid_pressure_fall", "feat_is_low_visibility", "feat_sky_obscured",
    "WeatherType_Rain", "WeatherType_Snow", "WeatherType_Fog",
    "WeatherType_Thunderstorm", "WeatherType_Drizzle", "WeatherType_Mist",
    "WeatherType_Haze", "WeatherType_Smoke", "WeatherType_Hail",
    "WeatherType_IcePellets",
]
 
# Categorical engineered cols — also use last() for the same reason
CATEGORICAL_ENGINEERED_COLS = [
    "feat_pressure_category", "feat_pressure_tendency_category",
    "feat_visibility_category", "feat_ceiling_category",
]
 
# DOT flight columns to carry forward — excludes actuals (DEP_TIME, ARR_TIME etc.)
# to avoid accidentally leaking outcome information into feature engineering
FLIGHTS_COLS = [
    "QUARTER", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "FL_DATE",
    "OP_UNIQUE_CARRIER", "OP_CARRIER_AIRLINE_ID", "OP_CARRIER",
    "TAIL_NUM", "OP_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID", "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID",
    "ORIGIN", "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR", "ORIGIN_STATE_FIPS",
    "ORIGIN_STATE_NM", "ORIGIN_WAC",
    "DEST_AIRPORT_ID", "DEST_AIRPORT_SEQ_ID", "DEST_CITY_MARKET_ID",
    "DEST", "DEST_CITY_NAME", "DEST_STATE_ABR", "DEST_STATE_FIPS",
    "DEST_STATE_NM", "DEST_WAC",
    "CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME", "DEP_DELAY", "DEP_DELAY_NEW",  # added CRS_ARR_TIME
    "DEP_DEL15", "DEP_DELAY_GROUP",
    "DISTANCE", "DISTANCE_GROUP", "YEAR",
]

# 2. Flights (`df_flights`)

- Read in
- Look at shape (num_rows, num_cols)
- filter 2015–2019, deduplicate (source data contains exact row duplicates), select relevant columns

In [0]:
df_flights = (
    spark.read.parquet(FLIGHTS_PATH)
    .filter(F.col("YEAR") < YEAR_MAX)
    .distinct()
    .select(FLIGHTS_COLS)
)
 
print(f"Rows: {df_flights.count():,} | Columns: {len(df_flights.columns)}")
df_flights.printSchema()

Rows: 31,746,841 | Columns: 38
root
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_AIRLINE_ID: integer (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- ORIGIN_CITY_MARKET_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- ORIGIN_STATE_FIPS: integer (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- ORIGIN_WAC: integer (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST_AIRPORT_SEQ_ID: integer (nullable

## 2.1 - Create table of unique airports (`df_airports`)

Built from unique ORIGIN/DEST pairs in `df_flights`, enriched with coordinates from `airport-codes_csv.csv` and `CITY_MARKET_ID` from the DOT airport dictionary.

> **Note on CITY_MARKET_ID**: 57 smaller regional airports are absent from the DOT dictionary. 5 inherited a market ID from a same-city airport; the remaining 52 use their own `AIRPORT_ID` as a placeholder. `CITY_MARKET_ID` is not a model feature — it is only used for market-level analysis.

In [0]:
# Pulling from ORIGIN and DEST separately then unioning ensures we capture
# every airport that ever appeared, regardless of whether it was more often
# an origin or a destination in the data
df_airports_origin = df_flights.select(
    F.col("ORIGIN_AIRPORT_ID").alias("AIRPORT_ID"),
    F.col("ORIGIN").alias("AIRPORT_CODE"),
    F.col("ORIGIN_CITY_NAME").alias("CITY_NAME"),
    F.col("ORIGIN_STATE_ABR").alias("STATE_ABR"),
    F.col("ORIGIN_STATE_FIPS").alias("STATE_FIPS"),
    F.col("ORIGIN_STATE_NM").alias("STATE_NM"),
    F.col("ORIGIN_WAC").alias("WAC"),
)
 
df_airports_dest = df_flights.select(
    F.col("DEST_AIRPORT_ID").alias("AIRPORT_ID"),
    F.col("DEST").alias("AIRPORT_CODE"),
    F.col("DEST_CITY_NAME").alias("CITY_NAME"),
    F.col("DEST_STATE_ABR").alias("STATE_ABR"),
    F.col("DEST_STATE_FIPS").alias("STATE_FIPS"),
    F.col("DEST_STATE_NM").alias("STATE_NM"),
    F.col("DEST_WAC").alias("WAC"),
)
 
df_airports = df_airports_origin.unionByName(df_airports_dest).distinct()
print(f"Unique airports: {df_airports.count():,}")

Unique airports: 372


In [0]:
# Also peek at the flight_airport_dictionary since that might be useful too
df_flight_airport_dict = spark.read.parquet("dbfs:/mnt/mids-w261/flight_airport_dictionary.parquet/")
print(f"Rows: {df_flight_airport_dict.count():,} | Columns: {len(df_flight_airport_dict.columns)}")
df_flight_airport_dict.printSchema()

# Enrich with CITY_MARKET_ID from DOT airport dictionary
df_airports = df_airports.join(
    df_flight_airport_dict.select("AIRPORT_ID", "CITY_MARKET_ID"),
    on="AIRPORT_ID",
    how="left"
)

Rows: 315 | Columns: 8
root
 |-- AIRPORT_ID: integer (nullable = true)
 |-- AIRPORT_SEQ_ID: integer (nullable = true)
 |-- CITY_MARKET_ID: integer (nullable = true)
 |-- ORIGIN/DEST: string (nullable = true)
 |-- CITY_NAME: string (nullable = true)
 |-- STATE_ABR: string (nullable = true)
 |-- STATE_NM: string (nullable = true)
 |-- WAC: integer (nullable = true)



In [0]:
# Also peek at the airport codes since that might be useful too
df_airport_codes = spark.read.csv(AIRPORT_CODES_PATH, header=True, inferSchema=True)
df_airport_codes.printSchema()

# Enrich df_airports with lat, long info
df_airports = (
    df_airports
    .join(
        df_airport_codes
        .filter(F.col("iso_country").isin(RELEVANT_COUNTRIES))
        .select(
            F.col("iata_code"),
            F.col("elevation_ft"),
            F.col("type").alias("airport_type"),
            F.col("coordinates"),
        ),
        on=df_airports["AIRPORT_CODE"] == F.col("iata_code"),
        how="left"
    )
    .drop("iata_code")
    .withColumn("LONGITUDE", F.split(F.col("coordinates"), ", ").getItem(0).cast("double"))
    .withColumn("LATITUDE",  F.split(F.col("coordinates"), ", ").getItem(1).cast("double"))
    .drop("coordinates")
)

root
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- coordinates: string (nullable = true)



In [0]:
# Check that all airports have coordinate info
df_airports.select(
    F.sum(F.when(F.col("LATITUDE").isNull(), 1).otherwise(0)).alias("missing_coords"),
    F.sum(F.when(F.col("elevation_ft").isNull(), 1).otherwise(0)).alias("missing_elevation"),
    F.count("*").alias("total")
).show()

+--------------+-----------------+-----+
|missing_coords|missing_elevation|total|
+--------------+-----------------+-----+
|             0|                1|  372|
+--------------+-----------------+-----+



# 3. Weather (`df_weather`)

- Read in 1yr of weather data that has been cleaned and had feature engineering done

In [0]:
# Read pre-processed weather — cleaning, imputation & feature engineering already done
df_weather = spark.read.parquet("dbfs:/student-groups/Group_01_02/kenzie_full_weather1.parquet")
print(f"Rows: {df_weather.count():,} | Columns: {len(df_weather.columns)}")

# Read pre-processed stations table — backup reconciliation & version IDs already done
df_all_stations = spark.read.parquet("dbfs:/student-groups/Group_01_02/kenzie_df_all_stations.parquet")
print(f"Stations: {df_all_stations.count():,}")

Rows: 529,888,509 | Columns: 78
Stations: 13,578


## 3.1 - Closest Weather Stations per Airport

Cross join airports × stations, compute haversine distance, keep top 5 closest per airport.

In [0]:
# Find the 5 closest weather stations to each airport using haversine distance:
#   - Cross join airports x stations to get all possible pairs
#       - Filter out stations with null coordinates before the cross join
#       - Alias lat/lon columns to avoid ambiguity
#   - Compute haversine distance in km for each pair
#   - Rank stations by distance per airport, keep top 5
#   - Pivot ranks into columns (closest_station_1..5 + distance)
#   - Join back to df_airports

# Cross join + distance + rank
df_cross = (
    df_airports
    .select(
        "AIRPORT_ID", "AIRPORT_CODE",
        F.col("LATITUDE").alias("airport_lat"),
        F.col("LONGITUDE").alias("airport_lon"),
    )
    .crossJoin(
        df_all_stations
        .filter(F.col("LATITUDE").isNotNull() & F.col("LONGITUDE").isNotNull())
        .select(
            F.col("STATION").alias("station_id"),
            F.col("LATITUDE").alias("station_lat"),
            F.col("LONGITUDE").alias("station_lon"),
        )
    )
    .withColumn("distance_km",
        F.lit(2) * F.lit(6371) * F.asin(F.sqrt(
            F.pow(F.sin(F.radians(F.col("station_lat") - F.col("airport_lat")) / 2), 2) +
            F.cos(F.radians(F.col("airport_lat"))) *
            F.cos(F.radians(F.col("station_lat"))) *
            F.pow(F.sin(F.radians(F.col("station_lon") - F.col("airport_lon")) / 2), 2)
        ))
    )
    .withColumn("rank", F.rank().over(Window.partitionBy("AIRPORT_ID").orderBy("distance_km")))
    .filter(F.col("rank") <= 5)
)

# Pivot top 5 stations into columns per airport
df_top5_pivot = (
    df_cross
    .groupBy("AIRPORT_ID")
    .agg(*[
        expr
        for i in range(1, 6)
        for expr in [
            F.max(F.when(F.col("rank") == i, F.col("station_id"))).alias(f"closest_station_{i}"),
            F.max(F.when(F.col("rank") == i, F.col("distance_km"))).alias(f"closest_station_{i}_km"),
        ]
    ])
)

df_airports = df_airports.join(df_top5_pivot, on="AIRPORT_ID", how="left")

# Sanity check: distance distribution for closest station
df_airports.select(
    F.min("closest_station_1_km").alias("min_km"),
    F.max("closest_station_1_km").alias("max_km"),
    F.avg("closest_station_1_km").alias("avg_km"),
    F.percentile_approx("closest_station_1_km", 0.5).alias("median_km"),
    F.percentile_approx("closest_station_1_km", 0.95).alias("p95_km"),
).show()

+--------------------+-----------------+------------------+------------------+------------------+
|              min_km|           max_km|            avg_km|         median_km|            p95_km|
+--------------------+-----------------+------------------+------------------+------------------+
|0.002915324989067...|89.21405849925497|1.7696805340393713|0.7013286434933705|4.6965964224061505|
+--------------------+-----------------+------------------+------------------+------------------+



## 3.2 - Time Windows

Join closest station info onto flights, then engineer the weather time window:
- `crs_dep_timestamp` = FL_DATE + CRS_DEP_TIME (rounded down to nearest 30 min)
- `prediction_time` = 1 hour before departure
- `weather_window` = [prediction_time − 3h, prediction_time]

In [0]:
# Add closest station columns from airport table
df_flights = (
    df_flights
    .join(
        df_airports.select(
            "AIRPORT_ID",
            "closest_station_1", "closest_station_1_km",
            "closest_station_2", "closest_station_2_km",
            "closest_station_3", "closest_station_3_km",
            "closest_station_4", "closest_station_4_km",
            "closest_station_5", "closest_station_5_km",
        ),
        on=df_flights["ORIGIN_AIRPORT_ID"] == F.col("AIRPORT_ID"),
        how="left"
    )
    .drop("AIRPORT_ID")
)

# 4. Engineer time window columns

In [0]:
# Engineer time window columns for weather join:
#   - Parse CRS_DEP_TIME (integer e.g. 945 → 09:45, 1315 → 13:15) into a timestamp
#       - Extract hour and minute via integer division and modulo
#       - Round minute down to nearest 30 (e.g. 47 → 30, 12 → 0)
#       - Combine with FL_DATE to build crs_dep_timestamp
#   - prediction_time = 1 hour before rounded departure
#       - Simulates the latest point at which a prediction could be made
#   - weather_window = [prediction_time - 3hrs, prediction_time]
#       - The 3-hour window of weather observations used as features
#   - flight_date = date portion of FL_DATE, used for partition pruning in the weather join
#   - Drop intermediate columns (dep_hour, dep_minute, dep_minute_rounded)
df_flights = (
    df_flights
    .withColumn("dep_hour",           (F.col("CRS_DEP_TIME") / 100).cast("int"))
    .withColumn("dep_minute",         (F.col("CRS_DEP_TIME") % 100).cast("int"))
    .withColumn("dep_minute_rounded", (F.floor(F.col("dep_minute") / 30) * 30).cast("int"))
    .withColumn("crs_dep_timestamp",
        F.to_timestamp(
            F.concat_ws(" ", F.col("FL_DATE"),
                F.lpad(F.col("dep_hour").cast("string"), 2, "0"),
                F.lpad(F.col("dep_minute_rounded").cast("string"), 2, "0"),
            ), "yyyy-MM-dd HH mm"
        )
    )
    .withColumn("prediction_time",      F.col("crs_dep_timestamp") - F.expr("INTERVAL 1 HOUR"))
    .withColumn("weather_window_start", F.col("prediction_time") - F.expr("INTERVAL 3 HOURS"))
    .withColumn("weather_window_end",   F.col("prediction_time"))
    .withColumn("flight_date",          F.to_date(F.col("FL_DATE")))
    .drop("dep_hour", "dep_minute", "dep_minute_rounded")
)

print(f"Rows: {df_flights.count():,}")
df_flights.select(
    "FL_DATE", "CRS_DEP_TIME", "crs_dep_timestamp",
    "prediction_time", "weather_window_start", "weather_window_end"
).show(5, truncate=False)

Rows: 31,746,841
+----------+------------+-------------------+-------------------+--------------------+-------------------+
|FL_DATE   |CRS_DEP_TIME|crs_dep_timestamp  |prediction_time    |weather_window_start|weather_window_end |
+----------+------------+-------------------+-------------------+--------------------+-------------------+
|2018-01-05|2356        |2018-01-05 23:30:00|2018-01-05 22:30:00|2018-01-05 19:30:00 |2018-01-05 22:30:00|
|2018-01-28|1042        |2018-01-28 10:30:00|2018-01-28 09:30:00|2018-01-28 06:30:00 |2018-01-28 09:30:00|
|2018-04-15|1707        |2018-04-15 17:00:00|2018-04-15 16:00:00|2018-04-15 13:00:00 |2018-04-15 16:00:00|
|2018-04-20|1445        |2018-04-20 14:30:00|2018-04-20 13:30:00|2018-04-20 10:30:00 |2018-04-20 13:30:00|
|2018-01-03|610         |2018-01-03 06:00:00|2018-01-03 05:00:00|2018-01-03 02:00:00 |2018-01-03 05:00:00|
+----------+------------+-------------------+-------------------+--------------------+-------------------+
only showing top 5 r

## 4.1 - Weather–Flight Join (`df_joined`)

Left join each flight to weather observations within its 3-hour window, aggregate to one row per flight.

- `obs_date.between(flight_date - 1, flight_date)` handles windows that cross midnight
- `try_cast` on weather columns handles non-numeric METAR codes (e.g. `VRB` for variable wind direction)
- Result: 31,746,841 rows — matches `df_flights.distinct()` confirming the join is correct

In [0]:
df_weather_slim = (
    df_weather
    .withColumn("obs_timestamp", F.to_timestamp(F.col("DATE"), "yyyy-MM-dd'T'HH:mm:ss"))
    .withColumn("obs_date", F.to_date(F.col("obs_timestamp")))
    .select(
        "STATION", "obs_timestamp", "obs_date",
        *NUMERIC_WEATHER_COLS,
        *NUMERIC_ENGINEERED_COLS,
        *BINARY_ENGINEERED_COLS,
        *CATEGORICAL_ENGINEERED_COLS,
    )
)

avg_exprs = (
    # Raw numeric cols — average across window
    [F.avg(F.expr(f"try_cast({c} as double)")).alias(f"avg_{c}")
     for c in NUMERIC_WEATHER_COLS] +
    # Numeric engineered cols — average across window
    [F.avg(F.col(c)).alias(f"avg_{c}")
     for c in NUMERIC_ENGINEERED_COLS] +
    # Binary flags + categorical — most recent non-null value in window
    [F.last(F.col(c), ignorenulls=True).alias(f"last_{c}")
     for c in BINARY_ENGINEERED_COLS + CATEGORICAL_ENGINEERED_COLS]
)

df_flights_r = df_flights.repartition(200, "closest_station_1")
df_weather_r = (
    df_weather_slim
    .withColumnRenamed("STATION", "wx_station")
    .repartition(200, "wx_station")
)

df_joined = (
    df_flights_r
    .join(
        df_weather_r,
        on=(
            (F.col("closest_station_1") == F.col("wx_station")) &
            (F.col("obs_date").between(
                F.date_sub(F.col("flight_date"), 1),
                F.col("flight_date")
            )) &
            (F.col("obs_timestamp") >= F.col("weather_window_start")) &
            (F.col("obs_timestamp") <= F.col("weather_window_end"))
        ),
        how="left"
    )
    .groupBy(*df_flights.columns)
    .agg(*avg_exprs)
)

print(f"Row count: {df_joined.count():,}")

Row count: 31,746,841


# 4. Graph

In [0]:
# Checkpoint df_joined before graph joins to break lineage and prevent
# duplicate column accumulation. Without this, Spark carries both sides
# of each join in its query plan, causing YEAR/MONTH/graph columns to
# appear multiple times when joins are re-run or chained.
CHECKPOINT_PATH = "dbfs:/student-groups/Group_01_02/df_joined_pre_graph.parquet"
df_joined.write.mode("overwrite").parquet(CHECKPOINT_PATH)
df_joined = spark.read.parquet(CHECKPOINT_PATH)
print(f"[8 pre] df_joined checkpointed and reloaded | "
      f"{df_joined.count():,} rows x {len(df_joined.columns)} cols")

[8 pre] df_joined checkpointed and reloaded | 31,746,841 rows x 99 cols


In [0]:
# This section adds four graph features that were not present in Phase 3.x:
#   - origin_pagerank:    how important is the origin airport in the network
#   - dest_pagerank:      same for the destination airport
#   - origin_betweenness: how often does the origin lie on shortest paths
#   - dest_betweenness:   same for the destination airport
#   - inbound_delay_flag: did the same aircraft arrive late on its inbound leg
#
# LEAKAGE RULE: All graph statistics are computed from the TRAINING partition
# only. Val and test rows receive the frozen train-era scores via a lookup join.
# This mirrors the same discipline applied to target-encoded features in Phase 3.3.
#
# Computing happens in two sub-steps:
#   A. Build graph snapshots in NetworkX from train data only (small enough for driver)
#   B. Self-join df_flights on TAIL_NUM for the cascade delay flag

# ── 8a. Temporal split — derive train partition for graph computation ──────────
#
# FIX (was): sort key was (MONTH * 100 + DAY_OF_MONTH), which ignores YEAR.
#   When data spans multiple calendar years, date values from different years
#   collapse into the same integers. The 60% cutoff then falls inside one year's
#   range, so entire years that land in val/test have NO rows in df_train_for_graph
#   → zero graph entries for those (airport, year, month) combos → 100% nulls on
#   test and partial nulls on val after the join.
#
# FIX (now): sort key is (YEAR * 10000 + MONTH * 100 + DAY_OF_MONTH) so the
#   ordering is globally chronological across all years.

df_joined_sorted = df_joined.withColumn(
    "_sort_date",
    (F.col("YEAR") * 10000 + F.col("MONTH") * 100 + F.col("DAY_OF_MONTH")).cast("int")
)
dates_sorted = (
    df_joined_sorted.select("_sort_date").distinct().orderBy("_sort_date").collect()
)
n = len(dates_sorted)
train_cutoff = dates_sorted[int(n * 0.6)]["_sort_date"]

# Train partition only — used exclusively for graph feature computation
df_train_for_graph = df_joined_sorted.filter(F.col("_sort_date") < train_cutoff)

print(f"[8a] Train partition for graph: {df_train_for_graph.count():,} rows | cutoff _sort_date={train_cutoff}")

# ── 8b. Compute monthly PageRank and betweenness centrality ──────────────────
#
# FIX: Graph stats are keyed on (AIRPORT, MONTH) only — not (AIRPORT, YEAR, MONTH).
#
# Rationale: joining on YEAR caused complete null coverage for any val/test year
# not represented in train (e.g. if train is 2015-2017 and test is 2019).
# Airport network topology is seasonal, not annual — a hub's centrality in
# January 2016 is a good proxy for January 2019. Dropping YEAR from the key:
#   1. Ensures every (airport, month) combo in val/test can find a match
#      as long as that airport appeared in ANY train month with the same MONTH.
#   2. Averages centrality across all train years for that month, which is
#      a more stable estimate than a single-year snapshot.
#
# Fallback: airports that appear in val/test but never in any train month
# receive the overall mean centrality (computed across all airports/months).
# This is better than leaving them null.

monthly_edges = (
    df_train_for_graph
    .groupBy("ORIGIN", "DEST", "MONTH")   # ← YEAR removed from group key
    .agg(F.count("*").alias("flight_count"))
    .toPandas()
)

print(f"[8b] Monthly edge table collected: {len(monthly_edges):,} rows | "
      f"{monthly_edges['MONTH'].nunique()} month snapshots")

graph_records = []

for month, group in monthly_edges.groupby("MONTH"):
    G = nx.DiGraph()
    for _, row in group.iterrows():
        G.add_edge(row["ORIGIN"], row["DEST"], weight=row["flight_count"])

    pr = nx.pagerank(G, alpha=0.85, weight="weight")
    bc = nx.betweenness_centrality(G, weight="weight", normalized=True)

    for airport in set(pr.keys()) | set(bc.keys()):
        graph_records.append({
            "AIRPORT":     airport,
            "MONTH":       int(month),   # ← no YEAR
            "pagerank":    pr.get(airport, 0.0),
            "betweenness": bc.get(airport, 0.0),
        })

    print(f"[8b]   month={int(month):02d}: {G.number_of_nodes()} airports, "
          f"{G.number_of_edges()} routes | "
          f"top PR={max(pr, key=pr.get)} ({max(pr.values()):.4f})")

df_graph_stats = spark.createDataFrame(pd.DataFrame(graph_records))

print(f"[8b] Graph stats DataFrame: {df_graph_stats.count():,} rows "
      f"({df_graph_stats.select('AIRPORT').distinct().count():,} unique airports, "
      f"12 months)")

# ── 8b-fallback. Compute per-airport global means for unseen airports ─────────
#
# Any airport that appears in val/test but never in ANY train month will still
# produce a null after the left join. We compute per-airport means across all
# months as a fallback, and impute those nulls afterward.

df_graph_means = (
    df_graph_stats
    .groupBy("AIRPORT")
    .agg(
        F.mean("pagerank").alias("pagerank_mean"),
        F.mean("betweenness").alias("betweenness_mean"),
    )
)

# Global fallback (single scalar) for airports completely absent from train
_global = df_graph_means.agg(
    F.mean("pagerank_mean").alias("g_pr"),
    F.mean("betweenness_mean").alias("g_bc"),
).collect()[0]
GLOBAL_PR_MEAN = float(_global["g_pr"])
GLOBAL_BC_MEAN = float(_global["g_bc"])
print(f"[8b-fallback] Global means — pagerank={GLOBAL_PR_MEAN:.6f}, betweenness={GLOBAL_BC_MEAN:.6f}")

# ── 8c. Join origin graph stats ───────────────────────────────────────────────
for c in ["origin_pagerank", "origin_betweenness"]:
    if c in df_joined.columns:
        df_joined = df_joined.drop(c)

df_joined = df_joined.join(
    df_graph_stats.select(
        F.col("AIRPORT").alias("_g_airport"),
        F.col("MONTH").alias("_g_month"),        # ← YEAR removed from join key
        F.col("pagerank").alias("origin_pagerank"),
        F.col("betweenness").alias("origin_betweenness"),
    ),
    on=(
        (F.col("ORIGIN") == F.col("_g_airport")) &
        (F.col("MONTH")  == F.col("_g_month"))
    ),
    how="left"
).drop("_g_airport", "_g_month")

# Impute remaining nulls (airports unseen in train) with airport-level mean,
# then fall back to global mean for airports with no train entry at all.
df_joined = (
    df_joined
    .join(
        df_graph_means.select(
            F.col("AIRPORT").alias("_fm_airport"),
            F.col("pagerank_mean").alias("_fm_pr"),
            F.col("betweenness_mean").alias("_fm_bc"),
        ),
        F.col("ORIGIN") == F.col("_fm_airport"),
        how="left"
    )
    .withColumn("origin_pagerank",    F.coalesce("origin_pagerank",    "_fm_pr",    F.lit(GLOBAL_PR_MEAN)))
    .withColumn("origin_betweenness", F.coalesce("origin_betweenness", "_fm_bc",    F.lit(GLOBAL_BC_MEAN)))
    .drop("_fm_airport", "_fm_pr", "_fm_bc")
)

_null_check = df_joined.select(
    F.sum(F.col("origin_pagerank").isNull().cast("int")).alias("null_origin_pr"),
    F.sum(F.col("origin_betweenness").isNull().cast("int")).alias("null_origin_bc"),
).collect()[0]
print(f"[8c] Origin graph stats joined | nulls remaining: "
      f"origin_pagerank={_null_check['null_origin_pr']}, "
      f"origin_betweenness={_null_check['null_origin_bc']}")

# ── 8d. Join dest graph stats ─────────────────────────────────────────────────
for c in ["dest_pagerank", "dest_betweenness"]:
    if c in df_joined.columns:
        df_joined = df_joined.drop(c)

df_joined = df_joined.join(
    df_graph_stats.select(
        F.col("AIRPORT").alias("_g_airport"),
        F.col("MONTH").alias("_g_month"),        # ← YEAR removed from join key
        F.col("pagerank").alias("dest_pagerank"),
        F.col("betweenness").alias("dest_betweenness"),
    ),
    on=(
        (F.col("DEST") == F.col("_g_airport")) &
        (F.col("MONTH") == F.col("_g_month"))
    ),
    how="left"
).drop("_g_airport", "_g_month")

# Same fallback pattern for dest
df_joined = (
    df_joined
    .join(
        df_graph_means.select(
            F.col("AIRPORT").alias("_fm_airport"),
            F.col("pagerank_mean").alias("_fm_pr"),
            F.col("betweenness_mean").alias("_fm_bc"),
        ),
        F.col("DEST") == F.col("_fm_airport"),
        how="left"
    )
    .withColumn("dest_pagerank",    F.coalesce("dest_pagerank",    "_fm_pr",    F.lit(GLOBAL_PR_MEAN)))
    .withColumn("dest_betweenness", F.coalesce("dest_betweenness", "_fm_bc",    F.lit(GLOBAL_BC_MEAN)))
    .drop("_fm_airport", "_fm_pr", "_fm_bc")
)

_null_check = df_joined.select(
    F.sum(F.col("dest_pagerank").isNull().cast("int")).alias("null_dest_pr"),
    F.sum(F.col("dest_betweenness").isNull().cast("int")).alias("null_dest_bc"),
).collect()[0]
print(f"[8d] Dest graph stats joined | nulls remaining: "
      f"dest_pagerank={_null_check['null_dest_pr']}, "
      f"dest_betweenness={_null_check['null_dest_bc']}")

# ── 8e. Tail-number cascade delay (self-join) ─────────────────────────────────

# Aircraft are reused across multiple flights in a day. If the inbound leg
# arrived late, the aircraft may not be ready for its next departure on time.
#
# LEAKAGE GUARD: We join on CRS_ARR_TIME (scheduled arrival), NOT ARR_TIME
# (actual arrival). At prediction time T-2h before the outbound departure,
# we do not yet know whether the inbound actually arrived late — but we do
# know its scheduled arrival time and the inbound flight's DEP_DEL15 label
# (which was known after the inbound departed, well before our prediction time).
#
# Join conditions:
#   - Same TAIL_NUM (same aircraft)
#   - Same FL_DATE (same calendar day)
#   - Inbound CRS_ARR_TIME is at least 30 minutes before outbound CRS_DEP_TIME
#     (30-minute buffer for turnaround — tighter than this is operationally unlikely)
#
# When multiple inbound legs exist for the same tail on the same day, we take
# the most recent one (max CRS_ARR_TIME) to get the leg that directly precedes
# the outbound departure.
df_inbound = (
    df_train_for_graph  # Only use train partition to avoid leakage
    .select(
        F.col("TAIL_NUM").alias("_ib_tail"),
        F.col("FL_DATE").alias("_ib_date"),
        F.col("CRS_DEP_TIME").alias("_ib_crs_arr"),
        F.col("DEP_DEL15").alias("_ib_dep_del15"),
    )
    # Keep only the most recent inbound leg per tail per day
    # to avoid fanout when one tail has multiple inbound legs
    .withColumn(
        "_ib_rank",
        F.row_number().over(
            Window.partitionBy("_ib_tail", "_ib_date")
            .orderBy(F.col("_ib_crs_arr").desc())
        )
    )
    .filter(F.col("_ib_rank") == 1)
    .drop("_ib_rank")
)

print(f"[8e] Inbound leg table built: {df_inbound.count():,} unique (tail, date) pairs")

df_joined = (
    df_joined
    .join(
        df_inbound,
        on=(
            (F.col("TAIL_NUM")     == F.col("_ib_tail"))  &
            (F.col("FL_DATE")      == F.col("_ib_date"))  &
            # Outbound must depart at least 30 minutes after inbound's scheduled arrival
            (F.col("CRS_DEP_TIME") >= F.col("_ib_crs_arr") + 30)
        ),
        how="left"
    )
    .withColumnRenamed("_ib_dep_del15", "inbound_delay_flag")
    .drop("_ib_tail", "_ib_date", "_ib_crs_arr")
    # Flights with no inbound leg (first flight of the day for this tail)
    # get 0 — not delayed by a predecessor
    .fillna(0, subset=["inbound_delay_flag"])
)

_cascade_counts = (
    df_joined
    .groupBy("inbound_delay_flag")
    .count()
    .orderBy("inbound_delay_flag")
    .collect()
)
print(f"[8e] Cascade join complete | inbound_delay_flag distribution:")
for row in _cascade_counts:
    print(f"       flag={int(row['inbound_delay_flag'])}: {row['count']:,} flights")

print(f"\n[8 DONE] df_joined shape: {df_joined.count():,} rows x {len(df_joined.columns)} cols")
print(f"         New graph columns: origin_pagerank, origin_betweenness, "
      f"dest_pagerank, dest_betweenness, inbound_delay_flag")


# 5. Checkpoint

In [0]:
df_joined.printSchema()

root
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_AIRLINE_ID: integer (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- ORIGIN_CITY_MARKET_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- ORIGIN_STATE_FIPS: integer (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- ORIGIN_WAC: integer (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- DEST_CITY_MARKET_

In [0]:
MERGED_PATH = "dbfs:/student-groups/Group_01_02/kenzie_full_df_joined.parquet"

df_joined.write.mode("overwrite").parquet(MERGED_PATH)
print(f"Saved df_joined → {MERGED_PATH}")
print(f"Rows: {df_joined.count():,}")
print(f"Cols: {len(df_joined.columns)}")

Saved df_joined → dbfs:/student-groups/Group_01_02/kenzie_full_df_joined.parquet
Rows: 31,746,841
Cols: 104


In [0]:
df_joined = spark.read.parquet(MERGED_PATH)
print(f"\nRead back rows: {df_joined.count():,}")
print(f"Read back cols: {len(df_joined.columns)}")
df_joined.printSchema()


Read back rows: 31,746,841
Read back cols: 104
root
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_AIRLINE_ID: integer (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- ORIGIN_CITY_MARKET_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- ORIGIN_STATE_FIPS: integer (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- ORIGIN_WAC: integer (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST_AIRPORT_SEQ_ID: 